# Taller 5: Ensamblaje de Genomas y Genómica Comparativa
## *Mycobacterium tuberculosis* vs. *Mycobacterium leprae*

**Pontificia Universidad Javeriana**  
**Facultad de Ingeniería — Departamento de Procesos**  
**Bioinformática para Ingenieros**  

* **Docente:** Msc. Juan Sebastián Malagón Torres  
* **Asistente de docencia:** Eliana Zuluaga Aponte  

---

### Objetivos de Aprendizaje
1. **Comprender los fundamentos del ensamblaje *de novo*:** Conocer cómo el algoritmo de grafos de De Bruijn y la estrategia multi-$k$ de SPAdes resuelven el rompecabezas a partir de lecturas cortas pareadas (*paired-end*).
2. **Ejecutar y parametrizar un pipeline de ensamblaje:** Automatizar el uso de **SPAdes** en aislados bacterianos controlando parámetros de hilos, corrección de errores y modos de ejecución.
3. **Evaluar rigurosamente la calidad genómica:**
   * **Calidad estructural (continuidad):** Medir N50, L50, longitud total y *misassemblies* con **QUAST**.
   * **Calidad biológica (completitud):** Evaluar la presencia de genes ortólogos conservados de copia única con **BUSCO** (`bacteria_odb10`).
4. **Aplicar genómica comparativa en Python:** Comparar dos patógenos milenarios del mismo género (*Mycobacterium*) para descubrir la huella evolutiva del colapso genómico y la pseudogenización masiva.

> [!NOTE]
> **Dinámica de la sesión (2 horas):**  
> * **Parte 1 (Yo Hago — Demostración Guiada):** El profesor ensambla y evalúa *Mycobacterium tuberculosis* (H37Rv).  
> * **Parte 2 (Tú Haces — Práctica Autónoma):** Cada estudiante/grupo ensambla y evalúa *Mycobacterium leprae* (TN).  
> * **Parte 3 (Comparamos Juntos — Genómica Comparativa):** Integramos ambos resultados en Python con visualizaciones avanzadas (Curva N50, Barras apiladas BUSCO y Dotplot sinténico 2D).


## 0. Preparación del Entorno y Verificación de Herramientas

Como siempre es un placer volver a verlos bioniños. Antes de comenzar a ensamblar, debemos asegurar que nuestro entorno cuente con los tres motores bioinformáticos que utilizaremos hoy:
* **SPAdes (`spades.py`):** Ensamblador *de novo* basado en grafos de De Bruijn multi-$k$.
* **QUAST (`quast.py`):** Evaluación de métricas estructurales de continuidad.
* **BUSCO (`busco`):** Benchmarking de genes ortólogos universales de copia única.

Ejecuta la siguiente celda para verificar la disponibilidad de las herramientas y cargar las librerías científicas en Python:

In [ ]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO

# Configuramos el estilo visual de los gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

# Aseguramos que el PATH priorice el entorno de bioinformática
os.environ["PATH"] = f"/home/jmalagont/miniconda3/envs/bioinformatics/bin:{os.environ.get('PATH', '')}"

print("=" * 60)
print("VERIFICACIÓN DE HERRAMIENTAS BIOINFORMÁTICAS")
print("=" * 60)

for cmd in ["spades.py --version", "quast.py --version", "busco --version"]:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    linea = (res.stdout or res.stderr).strip().split('\n')[0]
    print(f" [OK] {linea}")

print("=" * 60)
print("¡Entorno verificado y listo para ensamblar!")

### 0.1 Inspección de los Datos de Entrada

En la carpeta `data/` disponemos de los datos de secuenciación para ambos organismos:
* `data/Mycobacterium_tuberculosis/`:
  * `reads_R1.fastq.gz` y `reads_R2.fastq.gz` (Lecturas pareadas Illumina NextSeq 500, acceso `ERR17622119`).
  * `reference.fasta` (Genoma de referencia H37Rv, `NC_000962.3`, ~4.41 Mb).
* `data/Mycobacterium_leprae/`:
  * `reads_R1.fastq.gz` y `reads_R2.fastq.gz` (Lecturas pareadas Illumina HiSeq 2500, acceso `SRR6241768`).
  * `reference.fasta` (Genoma de referencia TN, `NC_002677.1`, ~3.27 Mb).

Revisemos el tamaño y peso de estos archivos en disco:

In [ ]:
!ls -lh data/Mycobacterium_tuberculosis/*.fastq.gz data/Mycobacterium_tuberculosis/*.fasta
!echo "--------------------------------------------------------"
!ls -lh data/Mycobacterium_leprae/*.fastq.gz data/Mycobacterium_leprae/*.fasta

---
## 1. Demostración Guiada ("Yo Hago"): *Mycobacterium tuberculosis*

*Mycobacterium tuberculosis* (cepa de referencia **H37Rv**) es el agente etiológico de la tuberculosis humana. Su genoma mide aproximadamente **4.41 Mb**, posee un contenido de Guanina-Citosina (GC) del **~65.6%** y codifica alrededor de **4.000 genes**.

### 1.1 ¿Cómo funciona SPAdes y por qué usamos la bandera `--isolate`?
1. **BayesHammer:** Antes de construir el grafo, aplica un modelo Bayesiano para detectar y corregir errores de secuenciación en las lecturas.
2. **Grafos de De Bruijn Multi-$k$:** A diferencia de ensambladores clásicos que usaban un solo valor de $k$ (ej. $k=31$), SPAdes construye grafos iterativos (típicamente $k=21, 33, 55$). Los valores pequeños de $k$ aportan conectividad en regiones poco cubiertas; los valores altos resuelven repeticiones cortas.
3. **Bandera `--isolate`:** Optimiza los parámetros de poda de burbujas (*bubble popping*) y cobertura asumiendo que el ADN proviene de un aislado clonal puro y no de un metagenoma diverso.

Lanzamos el ensamblaje para *M. tuberculosis*:

In [ ]:
# =========================================================================
# PARÁMETROS CONFIGURABLES: M. tuberculosis (Profesor)
# =========================================================================
MTB_R1 = "data/Mycobacterium_tuberculosis/reads_R1.fastq.gz"
MTB_R2 = "data/Mycobacterium_tuberculosis/reads_R2.fastq.gz"
OUT_SPADES_MTB = "resultados/assembly_mtb"
HILOS = 8

# Creamos la carpeta de resultados si no existe
os.makedirs("resultados", exist_ok=True)

# Comando de SPAdes
cmd_spades_mtb = f"spades.py -1 {MTB_R1} -2 {MTB_R2} --isolate -t {HILOS} -o {OUT_SPADES_MTB}"
print(f"Ejecutando: {cmd_spades_mtb}\n")

!{cmd_spades_mtb}

### 1.2 Inspección de los Contigs Resultantes

SPAdes genera dos archivos clave en la carpeta de salida:
* `contigs.fasta`: Secuencias contiguas construidas sin introducir huecos de inserción (*gaps* de Ns).
* `scaffolds.fasta`: Contigs ordenados y orientados usando la distancia conocida entre los extremos pareados (*paired-ends*), que pueden incluir Ns en las zonas no cubiertas.

Exploremos los contigs de *M. tuberculosis* en Python usando `BioPython`:

In [ ]:
contigs_mtb_file = f"{OUT_SPADES_MTB}/contigs.fasta"

# Leemos las longitudes de los contigs
longitudes_mtb = []
for record in SeqIO.parse(contigs_mtb_file, "fasta"):
    longitudes_mtb.append(len(record.seq))

longitudes_mtb.sort(reverse=True)

total_bases_mtb = sum(longitudes_mtb)
num_contigs_mtb = len(longitudes_mtb)
contig_max_mtb = longitudes_mtb[0]

print(f"Número total de contigs: {num_contigs_mtb}")
print(f"Longitud total ensamblada: {total_bases_mtb:,} pb")
print(f"Contig más largo: {contig_max_mtb:,} pb")
print(f"Top 5 contigs más largos (pb): {longitudes_mtb[:5]}")

### 1.3 Evaluación de Calidad Estructural con QUAST

Ahora evaluaremos la continuidad y la fidelidad del ensamblaje frente al genoma de referencia de *M. tuberculosis* (`NC_000962.3`).

> [!NOTE]
> **¿Qué es el N50?**  
> Si ordenas todos los contigs de mayor a menor y sumas sus longitudes, el **N50** es la longitud del contig en el cual la suma acumulada alcanza el **50% del tamaño total del ensamblaje**. Un N50 alto indica que la mayor parte del genoma está en piezas grandes y continuas.
> * **L50:** El número mínimo de contigs necesarios para alcanzar ese 50%.

Ejecutamos QUAST:

In [ ]:
REF_MTB = "data/Mycobacterium_tuberculosis/reference.fasta"
OUT_QUAST_MTB = "resultados/quast_mtb"

cmd_quast_mtb = f"quast.py {contigs_mtb_file} -R {REF_MTB} -o {OUT_QUAST_MTB} -t {HILOS}"
print(f"Ejecutando: {cmd_quast_mtb}\n")

!{cmd_quast_mtb}

In [ ]:
# Mostramos el reporte resumido de QUAST directamente en el notebook
quast_report_mtb = f"{OUT_QUAST_MTB}/report.txt"
with open(quast_report_mtb) as f:
    lineas = [l.strip() for l in f.readlines() if l.strip()]

# Imprimimos las métricas más relevantes
print("=" * 50)
print("MÉTRICAS CLAVE DE QUAST (M. tuberculosis)")
print("=" * 50)
for linea in lineas[:25]:
    print(linea)

### 1.4 Evaluación de Calidad Biológica con BUSCO

QUAST nos dice qué tan continuo es el ensamble, pero **no nos dice si los genes biológicamente esenciales están completos o rotos**. Para eso usamos **BUSCO** (*Benchmarking Universal Single-Copy Orthologs*).

BUSCO busca un conjunto de genes que deben estar presentes exactamente en una sola copia en todos los miembros del linaje bacteriano (`bacteria_odb10`, compuesto por **124 genes ortólogos universales**).

Clasifica cada gen en cuatro categorías:
* **Complete Single-copy (S):** Gen intacto presente una sola vez. (Ideal: porcentaje alto).
* **Complete Duplicated (D):** Gen completo pero con múltiples copias (puede sugerir duplicación real o quimerismo/sobre-ensamblado).
* **Fragmented (F):** Solo se ensambló un fragmento del gen (típico cuando el gen queda en el borde de un contig).
* **Missing (M):** El gen no fue encontrado en el ensamblaje.

Ejecutamos BUSCO para *M. tuberculosis*:

In [ ]:
OUT_BUSCO_MTB = "resultados/busco_mtb"

cmd_busco_mtb = f"busco -i {contigs_mtb_file} -m genome -l bacteria_odb10 -o {OUT_BUSCO_MTB} -f -c {HILOS}"
print(f"Ejecutando: {cmd_busco_mtb}\n")

!{cmd_busco_mtb}

### 1.5 Gráfico 1: Curva Acumulada de Contigs y Visualización del N50

Para entender visualmente qué significa el N50, graficaremos la longitud acumulada de los contigs ordenados de mayor a menor y trazaremos la línea del 50%:

In [ ]:
def graficar_curva_n50(contigs_fasta, titulo, color="#1f77b4"):
    # 1. Extraer y ordenar longitudes
    lens = sorted([len(rec.seq) for rec in SeqIO.parse(contigs_fasta, "fasta")], reverse=True)
    total_len = sum(lens)
    half_len = total_len / 2.0
    
    # 2. Calcular acumulados y N50
    acumulado = np.cumsum(lens)
    idx_n50 = np.where(acumulado >= half_len)[0][0]
    n50_val = lens[idx_n50]
    l50_val = idx_n50 + 1
    
    # 3. Crear el gráfico
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(range(1, len(lens) + 1), acumulado / 1e6, color=color, lw=2.5, label="Longitud acumulada")
    
    # Líneas de referencia para N50 y L50
    ax.axhline(half_len / 1e6, color="#d62728", ls="--", lw=1.5, label=f"50% del genoma ({half_len/1e6:.2f} Mb)")
    ax.axvline(l50_val, color="#2ca02c", ls=":", lw=1.5, label=f"L50 = {l50_val} contigs (N50 = {n50_val:,} pb)")
    
    ax.scatter([l50_val], [acumulado[idx_n50] / 1e6], color="#d62728", s=80, zorder=5)
    
    ax.set_title(f"Curva de Continuidad de Ensamblaje: {titulo}", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Número de contigs (ordenados de mayor a menor)", fontsize=11)
    ax.set_ylabel("Longitud acumulada (Mb)", fontsize=11)
    ax.legend(frameon=True, facecolor="white", loc="lower right")
    ax.grid(True, ls="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

graficar_curva_n50(contigs_mtb_file, "Mycobacterium tuberculosis (H37Rv)", color="#0077b6")

---
## 2. Práctica Autónoma ("Tú Haces"): *Mycobacterium leprae*

¡Ahora es su turno, bioniños!

Van a ensamblar y evaluar el genoma de ***Mycobacterium leprae*** (cepa de referencia **TN**), la bacteria causante de la lepra (enfermedad de Hansen).

### El Enigma Biológico de *M. leprae*
*M. tuberculosis* y *M. leprae* pertenecen al mismo género bacteriano, comparten un ancestro común cercano y poseen morfologías celulares muy similares. Sin embargo:
* *M. leprae* es un **parásito intracelular obligado** que solo puede replicarse dentro de células hospedadoras (macrófagos y células de Schwann).
* Debido a su aislamiento metabólico en el hospedero, *M. leprae* experimentó una **evolución reductiva drástica**: perdió genes que ya no necesitaba sintetizar por sí misma.
* Su genoma mide solo **~3.27 Mb** (frente a los 4.41 Mb de Mtb) y **más de la mitad de sus genes se convirtieron en pseudogenes** (secuencias de genes rotos o no funcionales).

---
### Misión del Estudiante:
1. Configurar las rutas y ejecutar SPAdes para *M. leprae*.
2. Evaluar el ensamble con QUAST usando `data/Mycobacterium_leprae/reference.fasta`.
3. Evaluar la completitud con BUSCO (`bacteria_odb10`).
4. Observar con atención el reporte de BUSCO: ¿Qué porcentaje de genes faltantes o fragmentados obtienes?

In [ ]:
# =========================================================================
# PARÁMETROS CONFIGURABLES: M. leprae (Estudiante)
# =========================================================================
# >>> TU CÓDIGO AQUÍ: Completa las rutas de los reads de M. leprae <<<
LEP_R1 = "data/Mycobacterium_leprae/reads_R1.fastq.gz"
LEP_R2 = "data/Mycobacterium_leprae/reads_R2.fastq.gz"
OUT_SPADES_LEP = "resultados/assembly_leprae"
HILOS = 8

# >>> TU CÓDIGO AQUÍ: Escribe el comando para ejecutar SPAdes con --isolate <<<
cmd_spades_lep = f"spades.py -1 {LEP_R1} -2 {LEP_R2} --isolate -t {HILOS} -o {OUT_SPADES_LEP}"
print(f"Ejecutando: {cmd_spades_lep}\n")

!{cmd_spades_lep}

### 2.2 Evaluación de Calidad Estructural de *M. leprae* con QUAST

In [ ]:
# =========================================================================
# EVALUACIÓN CON QUAST: M. leprae
# =========================================================================
# >>> TU CÓDIGO AQUÍ: Define la referencia y ejecuta QUAST para M. leprae <<<
REF_LEP = "data/Mycobacterium_leprae/reference.fasta"
OUT_QUAST_LEP = "resultados/quast_leprae"
CONTIGS_LEP = f"{OUT_SPADES_LEP}/contigs.fasta"

cmd_quast_lep = f"quast.py {CONTIGS_LEP} -R {REF_LEP} -o {OUT_QUAST_LEP} -t {HILOS}"
print(f"Ejecutando: {cmd_quast_lep}\n")

!{cmd_quast_lep}

### 2.3 Evaluación de Calidad Biológica de *M. leprae* con BUSCO

In [ ]:
# =========================================================================
# EVALUACIÓN CON BUSCO: M. leprae
# =========================================================================
# >>> TU CÓDIGO AQUÍ: Ejecuta BUSCO sobre los contigs de M. leprae <<<
OUT_BUSCO_LEP = "resultados/busco_leprae"

cmd_busco_lep = f"busco -i {CONTIGS_LEP} -m genome -l bacteria_odb10 -o {OUT_BUSCO_LEP} -f -c {HILOS}"
print(f"Ejecutando: {cmd_busco_lep}\n")

!{cmd_busco_lep}

In [ ]:
# Graficamos la curva de continuidad N50 de M. leprae
graficar_curva_n50(CONTIGS_LEP, "Mycobacterium leprae (TN)", color="#e76f51")

---
## 3. Genómica Comparativa en Python ("Comparamos Juntos")

¡Llegó la hora de integrar ambos resultados! Como futuros ingenieros bioinformáticos, no nos limitamos a correr comandos aislados: ahora procesaremos las salidas para responder preguntas biológicas profundas.

### 3.1 Comparación de Tamaño Total y Contenido de Guanina-Citosina (%GC)

En bacterias de vida libre como *M. tuberculosis*, los genomas son ricos en GC debido a presiones selectivas y mecanismos de reparación del ADN. En bacterias que sufren pseudogenización masiva como *M. leprae*, los genes desactivados acumulan mutaciones de transición que sesgan la composición hacia Adenina-Timina (AT).

Veamos si nuestros ensamblajes reflejan este fenómeno:

In [ ]:
def calcular_metricas_fasta(fasta_file):
    total_len = 0
    gc_count = 0
    num_seqs = 0
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq = str(record.seq).upper()
        total_len += len(seq)
        gc_count += seq.count("G") + seq.count("C")
        num_seqs += 1
    gc_pct = (gc_count / total_len * 100) if total_len > 0 else 0
    return total_len, gc_pct, num_seqs

len_mtb, gc_mtb, n_mtb = calcular_metricas_fasta(contigs_mtb_file)
len_lep, gc_lep, n_lep = calcular_metricas_fasta(CONTIGS_LEP)

df_comparativo = pd.DataFrame({
    "Métrica": ["Longitud Ensamblada (Mb)", "Contenido GC (%)", "Número de Contigs"],
    "M. tuberculosis": [round(len_mtb / 1e6, 3), round(gc_mtb, 2), n_mtb],
    "M. leprae": [round(len_lep / 1e6, 3), round(gc_lep, 2), n_lep],
    "Diferencia (Mtb - Leprae)": [
        round((len_mtb - len_lep) / 1e6, 3),
        round(gc_mtb - gc_lep, 2),
        n_mtb - n_lep
    ]
})

display(df_comparativo)

### 3.2 Gráfico 2: Dashboard Comparativo (Tamaño de Genoma vs %GC)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

organismos = ["M. tuberculosis", "M. leprae"]
colores = ["#0077b6", "#e76f51"]

# Subgráfico 1: Tamaño del genoma
axes[0].bar(organismos, [len_mtb / 1e6, len_lep / 1e6], color=colores, width=0.5, edgecolor="#333333")
axes[0].set_title("Tamaño Total del Ensamblaje (Mb)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Megabases (Mb)", fontsize=11)
axes[0].set_ylim(0, 5.0)
for i, v in enumerate([len_mtb / 1e6, len_lep / 1e6]):
    axes[0].text(i, v + 0.1, f"{v:.2f} Mb", ha="center", fontweight="bold", fontsize=10)

# Subgráfico 2: Contenido GC
axes[1].bar(organismos, [gc_mtb, gc_lep], color=colores, width=0.5, edgecolor="#333333")
axes[1].set_title("Contenido de Guanina-Citosina (%GC)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Porcentaje (% GC)", fontsize=11)
axes[1].set_ylim(0, 80)
for i, v in enumerate([gc_mtb, gc_lep]):
    axes[1].text(i, v + 1.5, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=10)

plt.suptitle("Impacto de la Evolución Reductiva en Mycobacterium", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

### 3.3 Gráfico 3: Comparativa Visual de BUSCO (Barras Apiladas)

Extraigamos automáticamente los resultados de los archivos de resumen de BUSCO (`short_summary*.txt`) y construyamos el gráfico de barras horizontales apiladas característico de la herramienta:

In [ ]:
import glob
import re

def parse_busco_summary(busco_dir):
    summary_files = glob.glob(f"{busco_dir}/short_summary.*.txt")
    if not summary_files:
        raise FileNotFoundError(f"No se encontró resumen en {busco_dir}")
    
    with open(summary_files[0]) as f:
        text = f.read()
    
    # Extraer porcentajes mediante expresiones regulares
    c = float(re.search(r"C:([0-9.]+)%", text).group(1))
    s = float(re.search(r"S:([0-9.]+)%", text).group(1))
    d = float(re.search(r"D:([0-9.]+)%", text).group(1))
    f_val = float(re.search(r"F:([0-9.]+)%", text).group(1))
    m = float(re.search(r"M:([0-9.]+)%", text).group(1))
    
    return {"Complete_Single": s, "Complete_Duplicated": d, "Fragmented": f_val, "Missing": m}

busco_mtb_data = parse_busco_summary(OUT_BUSCO_MTB)
busco_lep_data = parse_busco_summary(OUT_BUSCO_LEP)

df_busco = pd.DataFrame([busco_mtb_data, busco_lep_data], index=["M. tuberculosis", "M. leprae"])

# Gráfico de barras apiladas
fig, ax = plt.subplots(figsize=(10, 4))

categorias = ["Complete_Single", "Complete_Duplicated", "Fragmented", "Missing"]
nombres_legenda = ["Completo (Copia única)", "Completo (Duplicado)", "Fragmentado", "Ausente"]
colores_busco = ["#2a9d8f", "#457b9d", "#e9c46a", "#e63946"]

left = np.zeros(len(df_busco))
for cat, label, color in zip(categorias, nombres_legenda, colores_busco):
    valores = df_busco[cat].values
    ax.barh(df_busco.index, valores, left=left, color=color, label=label, height=0.45, edgecolor="white")
    
    # Etiquetas de porcentaje dentro de las barras
    for idx, (val, l) in enumerate(zip(valores, left)):
        if val > 3.0:
            ax.text(l + val / 2, idx, f"{val:.1f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=9)
    left += valores

ax.set_xlim(0, 100)
ax.set_xlabel("% de Genes Ortólogos Universales (bacteria_odb10, n=124)", fontsize=11)
ax.set_title("Comparación de Completitud Biológica (BUSCO)", fontsize=13, fontweight="bold", pad=12)
ax.legend(bbox_to_anchor=(0.5, -0.22), loc="upper center", ncol=4, frameon=True)
plt.tight_layout()
plt.show()

### 3.4 Gráfico 4: Dotplot Sinténico 2D (Colapso Genómico y Reorganizaciones)

Un **Dotplot** compara dos secuencias genómicas bidimensionalmente: el eje X representa una secuencia y el eje Y representa la otra. Si ambas secuencias son idénticas y colineales, se forma una línea diagonal perfecta. Si hay inversiones, la diagonal cambia de pendiente; y si hay deleciones masivas (pérdida de genes), la diagonal se interrumpe dejando espacios en blanco.

Comparemos el contig mayor de *M. tuberculosis* contra el contig mayor de *M. leprae* buscando $k$-mers compartidos:

In [ ]:
def generar_dotplot_kmers(seq1, seq2, k=15, step=1):
    # Diccionario de k-mers para la secuencia 1
    kmers_seq1 = {}
    for i in range(0, len(seq1) - k + 1, step):
        kmer = seq1[i:i+k]
        kmers_seq1.setdefault(kmer, []).append(i)
        
    x_points = []
    y_points = []
    
    # Buscar coincidencias en la secuencia 2
    for j in range(0, len(seq2) - k + 1, step):
        kmer = seq2[j:j+k]
        if kmer in kmers_seq1:
            for x_pos in kmers_seq1[kmer]:
                x_points.append(x_pos)
                y_points.append(j)
                
    return x_points, y_points

# Tomamos el contig más largo de cada ensamble
record_mtb_top = sorted(SeqIO.parse(contigs_mtb_file, "fasta"), key=lambda r: len(r.seq), reverse=True)[0]
record_lep_top = sorted(SeqIO.parse(CONTIGS_LEP, "fasta"), key=lambda r: len(r.seq), reverse=True)[0]

seq_mtb_sample = str(record_mtb_top.seq)[:30000]
seq_lep_sample = str(record_lep_top.seq)[:30000]

print(f"Comparando fragmentos sinténicos de 30 kb (k=13)...")
x_pts, y_pts = generar_dotplot_kmers(seq_mtb_sample, seq_lep_sample, k=13, step=2)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(x_pts, y_pts, s=1.0, color="#1d3557", alpha=0.6)
ax.set_title("Dotplot Sinténico 2D: M. tuberculosis vs. M. leprae", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel(f"M. tuberculosis ({record_mtb_top.id[:20]}...) [pb]", fontsize=10)
ax.set_ylabel(f"M. leprae ({record_lep_top.id[:20]}...) [pb]", fontsize=10)
ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 4. Preguntas Formativas y Pensamiento Crítico

Responde a las siguientes preguntas en celdas de Markdown justificando tus respuestas con los resultados obtenidos en este taller:

### Pregunta 1: Métricas Estructurales vs. Biológicas
> Un colega investigador afirma: *"Mi ensamblaje bacteriano tiene un N50 de 1.2 Mb, por lo tanto, es biológicamente completo y confiable"*.  
> **A la luz de lo aprendido hoy con QUAST y BUSCO, ¿por qué esta afirmación es incorrecta o incompleta?**

### Pregunta 2: El Dilema de $k$ en el Algoritmo de De Bruijn
> El género *Mycobacterium* contiene numerosas secuencias repetitivas como familias de genes PE/PPE y transposones.  
> **¿Qué ocurriría si intentaras ensamblar estas bacterias utilizando únicamente un valor pequeño de $k$ (por ejemplo, $k=21$)? ¿Cómo lo mitiga SPAdes con su enfoque multi-$k$?**

### Pregunta 3: El Caso de la Lepra: ¿Mala Calidad o Biología Real?
> Al correr BUSCO en *M. leprae*, encontramos una tasa de genes ausentes/fragmentados significativamente más alta que en *M. tuberculosis*.  
> **¿Significa esto que nuestro ensamblaje de M. leprae falló o tiene baja cobertura? ¿Cómo se explica este resultado desde la evolución reductiva y la pseudogenización?**

---
*Pontificia Universidad Javeriana — Bioinformática para Ingenieros*  
*Elaborado por: Juan Sebastián Malagón Torres, Eliana Zuluaga Aponte*
